# Exploratory data analysis

The goal of this notebook is to profile the data and see if any data cleaning or pre processing steps are required.

## Bronze to Silver - Data cleaning and basic pre-processing

Core idea of this layer: adress all the potential issues flagged by the data profiling library.

In [1]:
import os
import sys
import time
sys.path.append("..")

import pandas as pd
import polars as pl
from data_profiling import ProfileReport
from ingest import get_engine

/Users/aleksandrmedvedev/Desktop/Repositories/olist_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
engine = get_engine()

print("Waking up Azure SQL...")
for attempt in range(1, 11):
    try:
        with engine.connect() as conn:
            conn.exec_driver_sql("SELECT 1")
        print("Database is ready.")
        break
    except Exception:
        print(f"Attempt {attempt}/10 — retrying in 10s...")
        time.sleep(10)
else:
    raise RuntimeError("Database did not respond after 10 attempts.")

Waking up Azure SQL...
Attempt 1/10 — retrying in 10s...
Attempt 2/10 — retrying in 10s...
Database is ready.


In [3]:
customers            = pl.read_database("SELECT * FROM olist_customers_dataset", engine)
geolocation          = pl.read_database("SELECT * FROM olist_geolocation_dataset", engine)
order_items          = pl.read_database("SELECT * FROM olist_order_items_dataset", engine)
order_payments       = pl.read_database("SELECT * FROM olist_order_payments_dataset", engine)
order_reviews        = pl.read_database("SELECT * FROM olist_order_reviews_dataset", engine)
orders               = pl.read_database("SELECT * FROM olist_orders_dataset", engine)
products             = pl.read_database("SELECT * FROM olist_products_dataset", engine)
sellers              = pl.read_database("SELECT * FROM olist_sellers_dataset", engine)
category_translation = pl.read_database("SELECT * FROM product_category_name_translation", engine)

In [4]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation,
}

In [5]:
def generate_table_info(datasets, info_dir="info"):
    os.makedirs(info_dir, exist_ok=True)

    if os.listdir(info_dir):
        print(f"'{info_dir}' is not empty, skipping generation.")
        return

    tables_summary = []

    for name, df in datasets.items():
        lines = [
            f"Dataset: {name}",
            f"Shape: {df.shape[0]} rows x {df.shape[1]} columns",
            "Columns:",
            *(f"  - {col} ({dtype})" for col, dtype in zip(df.columns, df.dtypes)),
        ]
        content = "\n".join(lines)

        with open(f"{info_dir}/{name}.txt", "w") as f:
            f.write(content + "\n")

        tables_summary.append(content)
        print(f"Generated: {info_dir}/{name}.txt")

    with open(f"{info_dir}/tables.txt", "w") as f:
        f.write("\n\n".join(tables_summary) + "\n")

    print(f"Generated: {info_dir}/tables.txt")


generate_table_info(datasets)

Generated: info/customers.txt
Generated: info/geolocation.txt
Generated: info/order_items.txt
Generated: info/order_payments.txt
Generated: info/order_reviews.txt
Generated: info/orders.txt
Generated: info/products.txt
Generated: info/sellers.txt
Generated: info/category_translation.txt
Generated: info/tables.txt


In [6]:
dataset_reports = {}

for name, df in datasets.items():
    report = ProfileReport(df.to_pandas(), title=f"{name} profiling report", minimal=True, progress_bar=False) # minimal generation does not account for duplicates 
    report.to_file(f"html_reports/{name}.html")
    dataset_reports[name] = report
    print(f"Generated: html_reports/{name}.html")

100%|██████████| 5/5 [00:00<00:00, 11.72it/s]


Generated: html_reports/customers.html


100%|██████████| 5/5 [00:00<00:00, 10.36it/s]


Generated: html_reports/geolocation.html


100%|██████████| 7/7 [00:00<00:00, 16.90it/s]


...

Generated: html_reports/products.html


100%|██████████| 4/4 [00:00<00:00, 657.05it/s]


Generated: html_reports/sellers.html


100%|██████████| 2/2 [00:00<00:00, 29433.71it/s]


Generated: html_reports/category_translation.html


In [13]:
# TODO: implement transformations based on profiling report alerts
customers_cleaned            = customers
geolocation_cleaned          = geolocation
order_items_cleaned          = order_items
order_payments_cleaned       = order_payments
order_reviews_cleaned        = order_reviews
orders_cleaned               = orders
products_cleaned             = products
sellers_cleaned              = sellers
category_translation_cleaned = category_translation

In [ ]:
cleaned_datasets = {
    "customers": customers_cleaned,
    "geolocation": geolocation_cleaned,
    "order_items": order_items_cleaned,
    "order_payments": order_payments_cleaned,
    "order_reviews": order_reviews_cleaned,
    "orders": orders_cleaned,
    "products": products_cleaned,
    "sellers": sellers_cleaned,
    "category_translation": category_translation_cleaned,
}

In [ ]:
for name, df in cleaned_datasets.items():
    before = dataset_reports[name]
    after  = ProfileReport(df.to_pandas(), title=f"{name} - after", minimal=True, progress_bar=False) # minimal generation does not account for duplicates
    comparison = before.compare(after)
    comparison.to_file(f"html_reports/comparison_reports/{name}_cleaned_comparison.html")
    print(f"Generated: html_reports/comparison_reports/{name}_cleaned_comparison.html")

100%|██████████| 5/5 [00:00<00:00, 10.83it/s]


Generated: html_reports/comparison_reports/customers_cleaned_comparison.html


100%|██████████| 5/5 [00:00<00:00, 10.84it/s]


Generated: html_reports/comparison_reports/geolocation_cleaned_comparison.html


100%|██████████| 7/7 [00:00<00:00, 14.75it/s]


...

Generated: html_reports/comparison_reports/products_cleaned_comparison.html


100%|██████████| 4/4 [00:00<00:00, 479.77it/s]


Generated: html_reports/comparison_reports/sellers_cleaned_comparison.html


100%|██████████| 2/2 [00:00<00:00, 6825.56it/s]


Generated: html_reports/comparison_reports/category_translation_cleaned_comparison.html


## Silver to Gold - Reshaping the data for business analysis (Power BI)

Core idea of this layer: remove data redundancy and reshape the data into a star schema to support the business analysis.